In [1]:
!nvidia-smi

Fri Oct 30 23:54:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q vllm
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [3]:
!python -c "import torch; print(torch.__version__, '| CUDA', torch.version.cuda, '| GPU ok:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"

2.13.0+cu130 | CUDA 13.0 | GPU ok: True | Tesla T4


In [4]:
!git clone https://github.com/VatsalRoy/keep-it-warm.git
%cd /kaggle/working/keep-it-warm
!python -m env.generate_world --seed 42

fatal: destination path 'keep-it-warm' already exists and is not an empty directory.
/kaggle/working/keep-it-warm
seed=42  world_sha256=4a5b9a1a8c73a5e39f83a7c00eb5d4ccafec0d2bf6912ce82e81f31f2f0dc419


In [5]:
import subprocess

cmd = [
    "vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507",
    "--dtype", "half",
    "--max-model-len", "16384",
    "--enable-prefix-caching",
    "--enable-prompt-tokens-details",
    "--gpu-memory-utilization", "0.90",
    "--port", "8000",
]
log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("started, pid", server.pid)

started, pid 498


In [6]:
import time, requests
for i in range(90):
    if server.poll() is not None:
        print("Server CRASHED - see log below"); break
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"READY after ~{i*10}s"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("Not ready after 15 min - see log below")
!tail -n 30 vllm.log

READY after ~190s
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /tokenize, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /detokenize, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /v1/models, Methods: GET
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /v1/systemone, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /ping, Methods: GET
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /ping, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /invocations, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /v1/chat/completions, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /v1/chat/completions/batch, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.py:195] Route: /v1/responses, Methods: POST
(APIServer pid=498) INFO 10-30 23:57:53 [launcher.

In [7]:
!python scripts/apc_smoke_test.py


| Request | Prompt tokens | Cached (usage) | Cache hits (/metrics) | TTFT (ms) |
|---|---|---|---|---|
| 1 cold: new prefix | 5553 | 0 | 0 | 14162.6 |
| 2 warm: same prefix | 5551 | 5520 | 5520 | 229.4 |
| 3 edited: 1 char changed at start | 5554 | 0 | 0 | 13273.9 |

saved results/apc_smoke_test.json


In [8]:
!grep -iE "backend|attention|eager|cuda graph" vllm.log | head -20
!pip show vllm | head -2

(EngineCore pid=537) INFO 10-30 23:55:21 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityC

In [9]:
!pwd
!ls -la vllm.log
!wc -l vllm.log
!python -c "import vllm; print('vllm', vllm.__version__)"

/kaggle/working/keep-it-warm
-rw-r--r-- 1 root root 70523 Oct 30 23:58 vllm.log
341 vllm.log
vllm 0.32.0


In [10]:
!head -n 80 vllm.log

(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347] 
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.32.0
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen3-4B-Instruct-2507
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:347] 
(APIServer pid=498) INFO 10-30 23:55:03 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen3-4B-Instruct-2507', 'enable_prompt_tokens_details': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507', 'dtype': 'half', 'max_model_len': 16384, 'gpu_memory_utilization': 0.9, 'enable_prefix_caching': True}
(APIServer pid=498) INFO 10-30 23:55:04 [model.py:733] Resolved architecture: Qwen3ForCausalLM
(APIServer pid=498) WARNING 10-30 23:55:04 [model.py:2498] Casting torch.bfloat16 to 